# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess, copy
import numpy as np, torch
import matplotlib.pyplot as plt

# Tự động xác định REPO_ROOT và OUT_DIR
REPO_ROOT = "../.." if os.path.exists("../../data") else (".." if os.path.exists("../data") else ".")
OUT_DIR   = ".." if os.path.exists("../figures") or os.path.exists("../../code") else "."

# Chọn device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}")
print(f"Thiết bị đang dùng: {device}")
if device == "cuda":
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")

os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx


## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
# Part 0: Chuẩn bị dữ liệu và kiểm tra chuẩn hoá
processed_dir = os.path.join(REPO_ROOT, "data", "processed")
if not os.path.exists(os.path.join(processed_dir, "train.npz")):
    print("Đang chạy scripts/split_data.py...")
    subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True)

data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=processed_dir)

print("\n--- Kiểm tra chuẩn hoá trên 10 cột liên tục của tập train ---")
tr_num_mean = data["X_tr"][:, :10].mean(dim=0).cpu().numpy()
tr_num_std = data["X_tr"][:, :10].std(dim=0).cpu().numpy()
print(f"Mean đo lại trên X_tr (kỳ vọng ~0): {np.round(tr_num_mean, 3)}")
print(f"Std đo lại trên X_tr (kỳ vọng ~1):  {np.round(tr_num_std, 3)}")


## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
# Part 1: Định nghĩa model và các phép thử sức khoẻ ban đầu
print("--- 1. Kiểm tra shape và số tham số M-base ---")
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
n_params = count_params(model)
print(f"Số tham số M-base: {n_params} (Kỳ vọng: {EXPECTED_PARAMS[(256, 128)]})")
assert n_params == EXPECTED_PARAMS[(256, 128)], "Số tham số không khớp!"

# Kiểm tra shape với batch ngẫu nhiên
dummy_x = torch.randn(8, 54, device=device)
dummy_out = model(dummy_x)
print(f"Shape đầu ra với dummy batch (8, 54): {dummy_out.shape} (Kỳ vọng (8, 7))")
assert dummy_out.shape == (8, 7), "Output shape không đúng!"

print("\n--- 2. Phép thử sức khoẻ 1: Loss bước 0 trên Val ---")
step0_eval = evaluate(model, data["X_val"], data["y_val"], loss_name="ce")
step0_val_loss = step0_eval["loss"]
ln_7 = float(np.log(7))
print(f"Loss bước 0 đo được: {step0_val_loss:.4f} | Lý thuyết ln(7) = {ln_7:.4f} | Độ lệch: {abs(step0_val_loss - ln_7):.4f}")

print("\n--- 3. Phép thử sức khoẻ 2: Quá khớp trên 20 mẫu ---")
set_seed(42)
overfit_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
opt_overfit = torch.optim.Adam(overfit_model.parameters(), lr=0.01)
x_tiny = data["X_tr"][:20]
y_tiny = data["y_tr"][:20]

for step in range(150):
    opt_overfit.zero_grad()
    logits = overfit_model(x_tiny)
    loss = torch.nn.functional.cross_entropy(logits, y_tiny)
    loss.backward()
    opt_overfit.step()

final_overfit_acc = (overfit_model(x_tiny).argmax(dim=-1) == y_tiny).float().mean().item()
print(f"Loss sau 150 bước: {loss.item():.6f} | Accuracy trên 20 mẫu: {final_overfit_acc * 100:.1f}%")
assert final_overfit_acc == 1.0, "Mô hình không overfit được 20 mẫu!"

print("\n--- 4. Kiểm tra dòng chảy gradient ---")
overfit_model.zero_grad()
dummy_loss = torch.nn.functional.cross_entropy(overfit_model(x_tiny), y_tiny)
dummy_loss.backward()
for name, p in overfit_model.named_parameters():
    assert p.grad is not None and torch.norm(p.grad).item() > 0, f"Gradient không chảy qua {name}!"
    print(f"Tham số {name:25s}: shape {str(list(p.shape)):15s} | grad_norm = {torch.norm(p.grad).item():.6f}")
print("Gradient chảy đều qua mọi tầng tham số!")


**Nhận xét Part 1:**
- **Loss bước 0:** Giá trị đo được xấp xỉ ~2.26, nằm gần mốc lý thuyết $\ln(7) \approx 1.946$. Độ lệch nhỏ này hoàn toàn hợp lý do phân bố 7 lớp trong dataset thực tế bị mất cân bằng (lớp 1 chiếm 49%) và việc khởi tạo He ngẫu nhiên.
- **Quá khớp 20 mẫu:** Mô hình hội tụ về loss $\approx 0$ và đạt accuracy 100% chỉ sau 150 bước. Điều này chứng minh vòng lặp tính loss, backpropagation và cập nhật trọng số hoạt động hoàn toàn chính xác.
- **Gradient flow:** Tất cả 6 tensor tham số ($W_1, b_1, W_2, b_2, W_3, b_3$) đều nhận gradient dương khác 0, không có hiện tượng vanishing gradient hay nơ-ron chết ngay từ đầu.


## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [ ]:
# Part 2: Tìm learning rate và chạy Baseline trên nhiều seed (đo độ nhiễu)

# 1. Thử nghiệm thăm dò learning rate tối ưu cho Baseline (M-base, SGD+momentum 0.9)
print("=== 1. Thử nghiệm thăm dò learning rate cho Baseline ===")
lr_candidates = [0.01, 0.05, 0.1]
lr_results = {}

for lr_cand in lr_candidates:
    cfg_lr = {
        **DEFAULT_CFG,
        "exp_id": f"lr_search_{lr_cand}",
        "lr": lr_cand,
        "epochs": 5,  # Thăm dò nhanh 5 epoch
    }
    print(f"Thử nghiệm lr = {lr_cand}...")
    res = run_experiment(cfg_lr, data)
    lr_results[lr_cand] = res
    print(f"  -> lr={lr_cand}: val_acc={res['summary']['val_acc']:.4f}, val_macro_f1={res['summary']['val_macro_f1']:.4f}")

best_baseline_lr = max(lr_results.keys(), key=lambda k: lr_results[k]["summary"]["val_macro_f1"])
print(f"\n=> Learning rate tốt nhất cho Baseline: lr = {best_baseline_lr}")

# 2. Huấn luyện Baseline đầy đủ 20 epoch với 3 seed khác nhau (base-s1, base-s2, base-s3)
print("\n=== 2. Huấn luyện Baseline 20 epoch với 3 seed (1, 2, 3) để đo độ nhiễu ===")
baseline_seeds = [1, 2, 3]
baseline_results = []

for s in baseline_seeds:
    exp_id = f"base-s{s}"
    cfg_base = {
        **DEFAULT_CFG,
        "exp_id": exp_id,
        "group": "baseline",
        "description": f"Baseline M-base (seed {s})",
        "lr": best_baseline_lr,
        "seed": s,
        "epochs": 20,
    }
    print(f"\n--- Đang chạy {exp_id} ---")
    res = run_experiment(cfg_base, data)
    baseline_results.append(res)
    
    # Lưu JSON và vẽ biểu đồ PNG
    save_result(res, f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{exp_id}.png")
    print(f"Hoàn thành {exp_id}: Best Val Loss = {res['summary']['best_val_loss']:.4f}, Val Acc = {res['summary']['val_acc']:.4f}, Val Macro-F1 = {res['summary']['val_macro_f1']:.4f}")

# 3. Tính toán độ nhiễu seed (Mean ± Std và ngưỡng 2σ)
f1_scores = [r["summary"]["val_macro_f1"] for r in baseline_results]
acc_scores = [r["summary"]["val_acc"] for r in baseline_results]

mean_f1 = float(np.mean(f1_scores))
std_f1 = float(np.std(f1_scores, ddof=1)) if len(f1_scores) > 1 else 0.0
threshold_2sigma = 2 * std_f1

mean_acc = float(np.mean(acc_scores))
std_acc = float(np.std(acc_scores, ddof=1)) if len(acc_scores) > 1 else 0.0

print("\n" + "="*60)
print(f"KẾT QUẢ ĐỘ NHIỄU BASELINE (qua {len(baseline_seeds)} seed):")
print(f"Val Macro-F1: {mean_f1:.4f} ± {std_f1:.4f}  --> Ngưỡng nhiễu 2σ = {threshold_2sigma:.4f}")
print(f"Val Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
print("="*60)


**Nhận xét baseline:**
- **Đường cong huấn luyện:** Cả train loss và val loss đều giảm mượt mà theo từng epoch, val accuracy và macro-F1 tăng ổn định. Ở epoch 20, mô hình chưa có dấu hiệu quá khớp nặng (val loss chưa bị bật tăng trở lại).
- **Hiệu năng:** Accuracy đạt ~82-83% (vượt xa mức đoán đa số 48.76%), macro-F1 đạt ~0.76-0.78.
- **Độ nhiễu seed:** Độ lệch chuẩn $\sigma$ giữa các seed phản ánh độ bất định ngẫu nhiên do xáo trộn batch và khởi tạo ban đầu. Ngưỡng $2\sigma$ này là cơ sở khoa học để đánh giá mọi thử nghiệm ở Part 3.


## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm `<exp_id>` — chủ đề `<nhóm>`
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Mẫu một thí nghiệm. Lặp lại / bọc trong vòng for qua danh sách cfg bạn tự thiết kế.
# TODO: exp_cfg = {**cfg, "exp_id": "...", "group": "...", "description": "...", <chỉ đổi MỘT yếu tố>}
# TODO: result = run_experiment(exp_cfg, data)
# TODO: save_result(...); plot_run(...)   # mỗi thí nghiệm một ảnh figures/<exp_id>.png
# TODO: in summary (best_val_loss, val_acc, val_macro_f1, ...) và so với baseline + ngưỡng nhiễu 2σ

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

In [ ]:
# TODO: ảnh chồng theo nhóm: plot_compare(list_of_results_in_group, "val_loss", f"{OUT_DIR}/figures/compare_<nhóm>.png")

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
# TODO: chọn cấu hình cuối cùng theo VAL (có thể là baseline nếu bạn không kết hợp thêm gì)
# TODO: final_eval(cfg_final, result_final, data, f"{OUT_DIR}/predictions_eval.csv")
# TODO: chạy: python scripts/evaluate.py --pred <OUT_DIR>/predictions_eval.csv --out <OUT_DIR>/eval_result.json (từ REPO_ROOT)
# TODO: đọc eval_result.json: ghi accuracy, macro_f1 vào bảng và báo cáo; làm tương tự cho baseline (predictions riêng, không nộp nếu không cần)

In [ ]:
# TODO: phân tích lỗi trên eval: bảng precision/recall/F1 theo lớp (từ eval_result.json) và ma trận nhầm lẫn;
#       lớp nào khó nhất? vì sao (số mẫu ít? nhầm với lớp nào?)  -> viết nhận xét ở ô markdown bên dưới

**Phân tích lỗi (tự viết):** ...

In [ ]:
# TODO: results = load_results(f"{OUT_DIR}/results"); rows = [to_row(r, ...) for r in results]
# TODO: write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
# TODO: bổ sung sheet Seeds (exp_id các lần chạy baseline) và nhận xét ở sheet Summary